# Panorama da Segurança Operacional nos Aeroportos do Brasil:
## Uma Análise Baseada em Ocorrências e Exposição (2023–2024)

A segurança operacional na aviação é tradicionalmente avaliada por meio da análise de ocorrências aeronáuticas, como acidentes e incidentes. No entanto, análises baseadas exclusivamente em números absolutos podem levar a interpretações equivocadas, uma vez que não consideram o volume de operações ao qual o sistema está exposto.

Segundo a International Civil Aviation Organization (ICAO, 2018), a avaliação da segurança deve ser realizada com base em indicadores normalizados por exposição, como taxas de ocorrências por número de voos ou horas de voo. Essa abordagem é amplamente adotada pela indústria, sendo utilizada, por exemplo, nos relatórios da International Air Transport Association (IATA), que expressam o desempenho de segurança em termos de acidentes por milhão de voos.

Este estudo tem por objetivo verificar a exposição ao risco dos aeroportos do Brasil, considerando o número de ocorrências comparado com o volume de operação de cada unidade, ao invés de observar números absolutos. Isso nos dá um índice de risco proporcional, que torna possível a comparação da segurança operacional entre aeroportos.

### Hipóteses avaliadas
Dado o contexto de segurança operacional apresentado anteriormente, buscaremos ao longo do estudo testar as seguintes hipóteses:

**H1:** O número de ocorrências aeronáuticas não cresce proporcionalmente ao volume de operações, apresentando comportamento sublinear quando ajustado por exposição.

**H2:** Incidentes apresentam maior concentração em determinados aeroportos, enquanto acidentes e incidentes graves possuem distribuição mais uniforme.

**H3:** A utilização de métricas baseadas em valores absolutos distorce a avaliação da segurança operacional, enquanto métricas normalizadas por volume de operações permitem comparações mais adequadas entre aeroportos.

### Dados Utilizados

Neste estudo são utilizados dados disponibilizados por dois órgãos nacionais responsáveis pela gestão do serviço aeroportuário do Brasil: A ANAC e o CENIPA. A ANAC é a agencia reguladora responsável por auditar e fiscalizar a aviação civil no país. O CENIPA é um órgão subordinado às Força Aerea Brasileira responsável pela investigação dos incidentes e acidentes. Cada investigação gera um relatório com um conjunto de recomendações que tem como objetivo evitar que ocorrências similares aconteçam.

Toda ocorrência no serviço aéreo em território brasileiro deve ser reportada ao CENIPA. Os dados são diponibilizados seguindo a política de dados abertos do Governo Federal. Os dados são disponibilizados em formato csv e atualizados na peridicidade de conveniência à Agência.

# Apresentação dos dados
Os dados foram obtidos diretamente do portal de dados abertos do Governo Brasileiro e os links diretos para as fontes podem ser observados no arquivo ```fontes de coleta.rtf```

Os números das principais variáveis avaliadas ao longo desse trabalho são resultado de cruzamentos dos dados:
- do **CENIPA/FAB** — base de ocorrências aeronáuticas (acidentes, incidentes graves e incidentes) registrada pelo Centro de Investigação e Prevenção de Acidentes Aeronáuticos desde 2007.
- da **ANAC** — dados de VRA (Voos Regulares Ativos) de 2023 e 2024, com granularidade de voo individual e situação de execução. Os dados de VRA são divulgados mês a mês.

## Carregamento dos dados

| DataFrame | Shape | Fonte | Encoding | Separador |
|---|---|---|---|---|
| `df_vra_2023` | 981 206 × 8 | 12 CSVs mensais VRA 2023 (jan–dez) | UTF-8 | `;` |
| `df_vra_2024` | 987 868 × 8 | 12 CSVs mensais VRA 2024 (jan–dez) | UTF-8 | `;` |
| `df_ocorrencia` | ~13 186 × 22 | `ocorrencia.csv` (CENIPA/FAB) | latin1 | `;` |
| `df_aeronave` | ~13301, 23 | `aeronave.csv` (CENIPA/FAB) | latin1 | `;` |


As 8 colunas retidas do VRA via usecols são: ICAO da empresa, nome da empresa, assentos, ICAO de origem, descrição do aeroporto de origem, ICAO de destino, descrição do aeroporto de destino e Situação Voo

In [1]:
repo_url = 'https://github.com/fmariane/sprint1-mvp.git'
!git clone {repo_url}

# ativando o caminho do repo sob o grupo files
import os
repo_name = repo_url.split('/')[-1].replace('.git', '')
os.chdir(repo_name)

if os.path.exists(repo_name):
    os.chdir(repo_name)
    print(f"Changed directory to: {os.getcwd()}")
else:
    print(f"Error: Repository directory '{repo_name}' not found. Did the clone operation complete successfully?")

Cloning into 'sprint1-mvp'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 155 (delta 19), reused 39 (delta 12), pack-reused 104 (from 1)
Receiving objects: 100% (155/155), 46.94 MiB | 29.20 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Error: Repository directory 'sprint1-mvp' not found. Did the clone operation complete successfully?


In [2]:
# load data into dataframes
import sys
from pathlib import Path

import pandas as pd
import plotly.express as ptex
import matplotlib.pyplot as plt
from os import path
from glob import glob

# allow `from utils ...` whether cwd is `source/` or project root
for _p in (Path.cwd(), Path.cwd() / "source"):
    if (_p / "utils" / "dataframe_operations.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from utils.dataframe_operations import (
    DataFrameOperations, load_datasource_urls, filter_by_year,
    build_ocorrencias_por_aeroporto,
)

datasources = load_datasource_urls()
loader = DataFrameOperations()

df_aeronave = loader.load_dataframe(datasources["aeronave"])
df_ocorrencia = loader.load_dataframe(datasources["ocorrencia"])

In [3]:
# carregamento dos dados do Registro de Voos ativos, anos 2023
# Dados sao recebidos em formato de arquivos separados por mes. Sao concatenados em um unico dataframe
# Considerando o shape desses dataframes, essa célula e a seguinte podem demorar ate 1 min para executar
cols=["Sigla ICAO Empresa Aérea","Empresa Aérea","Número de Assentos","Sigla ICAO Aeroporto Origem",
       "Descrição Aeroporto Origem","Sigla ICAO Aeroporto Destino","Descrição Aeroporto Destino","Situação Voo"]
df_vra_2023 = loader.load_dataframe(datasources["vra_2023_01"],
datasources["vra_2023_02"],
datasources["vra_2023_03"],
datasources["vra_2023_04"],
datasources["vra_2023_05"],
datasources["vra_2023_06"],
datasources["vra_2023_07"],
datasources["vra_2023_08"],
datasources["vra_2023_09"],
datasources["vra_2023_10"],
datasources["vra_2023_11"],
datasources["vra_2023_12"],
usecols=cols,encoding="utf-8")

In [4]:
df_vra_2024 = loader.load_dataframe(datasources["vra_2024_01"],
datasources["vra_2024_02"],
datasources["vra_2024_03"],
datasources["vra_2024_04"],
datasources["vra_2024_05"],
datasources["vra_2024_06"],
datasources["vra_2024_07"],
datasources["vra_2024_08"],
datasources["vra_2024_09"],
datasources["vra_2024_10"],
datasources["vra_2024_11"],
datasources["vra_2024_12"],
usecols=cols,encoding="utf-8")

# Analise do conjunto de dados de "Ocorrencias".
Na aviação, as ocorrências operacionais ou de segurança são classificadas em acidentes, incidentes graves e apenas incidentes. Os incidentes são aqueles em que a ocorrência afeta o funcionamente e compromete a segurança operacional, porém sem danos e sem feridos. Os incidentes graves envolvem uma falha séria de segurança, mas ainda sem a ocorrência de feridos. Os acidentes são ocorrências em que ocorre dano grave à aeronave e ou/feridos (CENIPA NSCA 3-13).
Vamos observar como estes dados estao dispostos por meio de algumas ferramentas de visualizalção.
Portanto vamos observar primeiramente a diferença da taxa de acidentes e incidentes registrados no dataset ocorrencias.

In [5]:
#TRATAMENTO DOS DADOS: Originalmente essa coluna era do tipo string, no entanto para melhor analisar e manipular os dados
# o tipo de dados da coluna foi alterado para datetime usando recursos da biblioteca pandas

#contagem simples de ocorrencias por ano e contagem segmentada por tipo de ocorrencia
from narwhals._compliant import group_by

df_ocorrencia['ocorrencia_dia'] = pd.to_datetime(
    df_ocorrencia['ocorrencia_dia'],
    format='%d/%m/%Y',
    dayfirst=True,
    errors='coerce'
)

#evolução dos regitros das ocorrências por ano
years = df_ocorrencia['ocorrencia_dia'].dt.year
count_ocorr_ano = years.value_counts().sort_index()

ocorrencias_por_ano = ptex.bar(count_ocorr_ano)
ocorrencias_por_ano.update_layout(width=500, height=500)
ocorrencias_por_ano.show()

df_valid = df_ocorrencia[df_ocorrencia['ocorrencia_dia'].notna()].copy()
df_valid['ano'] = df_valid['ocorrencia_dia'].dt.year

#Distribuição por ano de quantas ocorrencias foram acidentes e quantas ocorrências foram incidentes
grouped_counts = (
    df_valid
    .groupby(['ano', 'ocorrencia_classificacao'])
    .size()
)

tabela = grouped_counts.reset_index(name='contagem')
tabela['ano'] = tabela['ano'].astype(str)

ocorrencias_seg_ano = ptex.bar(
    tabela,
    x='ano',
    y='contagem',
    color='ocorrencia_classificacao',
    barmode='stack',
    labels={'ano': 'Ano', 'contagem': 'Número de ocorrências', 'ocorrencia_classificacao': 'Classificação'},
    title='Ocorrências por ano segmentadas por classificação',
    width=800,
    height=500,
     color_discrete_map={
        "INCIDENTE": "#27D3F5",
        "INCIDENTE GRAVE": "#f7a831",
        "ACIDENTE": "#EF553B",
    }
)
ocorrencias_seg_ano.show()

## Distribuição Espacial das Ocorrências
Considerando a variável numérica _quantidade de ocorrências_, representada nos gráficos acima, nota-se um aumento expressivo de volume nos anos de 2023 e 2024.
Por conta desse aumento significativo, os anos de 2023 e 2024 serão alvo direto deste estudo.

Conforme discutido na apresentação do problema, números absolutos não geram um indicativo forte para segurança operacional. Vamos observar a disposição espacial das ocorrências utilizando os atributos de latitude e longitude presentes no dataframe ```df_ocorrencia```

In [6]:
#distribuição espacial de ocorrencias por latitude e longitude, anos 2023 e 2024

import math
from utils.dataframe_operations import filter_by_year

df_ocorrencia_brasil = df_ocorrencia[
    df_ocorrencia['ocorrencia_pais'].str.upper() == 'BRASIL'
]

df_ocorrencia_brasil = filter_by_year(df_ocorrencia_brasil, 'ocorrencia_dia', [2023, 2024])

null_lat = pd.to_numeric(df_ocorrencia_brasil['ocorrencia_latitude'], errors='coerce').isna().sum()
null_lon = pd.to_numeric(df_ocorrencia_brasil['ocorrencia_longitude'], errors='coerce').isna().sum()
print(f"Nulos — latitude: {null_lat}, longitude: {null_lon}  (de {len(df_ocorrencia_brasil)} registros)")

lat_col = pd.to_numeric(df_ocorrencia_brasil["ocorrencia_latitude"], errors="coerce").dropna()
lon_col = pd.to_numeric(df_ocorrencia_brasil["ocorrencia_longitude"], errors="coerce").dropna()

lat_min, lat_max = lat_col.min(), lat_col.max()
lon_min, lon_max = lon_col.min(), lon_col.max()

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2
zoom = math.log2(360 / max(lat_max - lat_min, lon_max - lon_min)) - 1

dist_espacial_ocorrencias  = ptex.scatter_map(
    df_ocorrencia_brasil,
    lat="ocorrencia_latitude",
    lon="ocorrencia_longitude",
    hover_name="ocorrencia_cidade",
    color="ocorrencia_classificacao",
    category_orders={
        "ocorrencia_classificacao": ["ACIDENTE", "INCIDENTE GRAVE", "INCIDENTE"]
    },
    color_discrete_map={
        "INCIDENTE": "#27D3F5",
        "INCIDENTE GRAVE": "#f7a831",
        "ACIDENTE": "#EF553B",
    },
    zoom=3.3,
    center={"lat": -14, "lon": -52},
    map_style="carto-positron",
    opacity=0.6,
    title="Ocorrências Aéreas por Localização — Brasil"
)

counts = df_ocorrencia_brasil['ocorrencia_classificacao'].value_counts()
subtitle = "  ·  ".join(f"{cls}: {n}" for cls, n in counts.items())

dist_espacial_ocorrencias.update_layout(
    height=1300,
    width=1200,
    map=dict(
       # bounds=dict(west=-75, east=-34, south=-34, north=6)
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        title_text="",
        itemclick="toggle",
        itemdoubleclick="toggleothers"
    ),
    annotations=[dict(
        text=f"Total: {counts.sum()}  ·  {subtitle}",
        xref="paper", yref="paper",
        x=0.5, y=-0.03,
        showarrow=False,
        font=dict(size=13),
        xanchor="center",
        yanchor="top",
    )]
)

dist_espacial_ocorrencias.show()

Nulos — latitude: 231, longitude: 231  (de 4084 registros)


### Disposição da distribuição espacial

Infelizmente, a disposição espacial por si só não nos dá uma ideia clara de como acidentes e incidentes estão distribuídos, dado que normalmente muitos pontos estão sobrepostos se considerarmos que as ocorrências frequentemente acontecem em coordenadas próximas (áreas de aeroportos e aerodrómos). Apesar de não ser possível fazer uma afirmação categórica, percebe-se que _incidentes se sobrepoem com mais frequencia que os incidentes_ devido a maior opacidade dos pontos azuis. Partindo dessa premissa vamos manter em mente a possibilidade de que _acidentes tem uma distribuição mais uniforme enquanto incidentes tem uma distribuição concentrada em certos hubs/clusters_.

Pensando em confirmar essa ideia, vamos agrupar as ocorrências por cidade, assim poderemos ter mais clareza sobre os pontos sobrepostos

In [7]:
#contagem de ocorrencias, agrupadas por cidade e separadas por tipo de ocorrencia
from utils.dataframe_operations import filter_by_year

df_ocorrencia_recente = filter_by_year(df_ocorrencia, 'ocorrencia_dia', [2023, 2024])

df_acidentes_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'ACIDENTE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

df_incidentes_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'INCIDENTE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

df_incidentes_graves_cidade = (
    df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_classificacao'] == 'INCIDENTE GRAVE']
    .groupby('ocorrencia_cidade').size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

In [8]:
#subplots de ocorrencias por cidade, por classificacao e por ano
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

classificacoes = ['ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE']
anos = [2023, 2024]
cores = {'ACIDENTE': '#EF553B', 'INCIDENTE GRAVE': '#f7a831', 'INCIDENTE': '#27D3F5'}

dist_cidades_ano_classificacao = make_subplots(
    rows=len(anos), cols=len(classificacoes),
    vertical_spacing=0.15,
    horizontal_spacing=0.08,
)

for row_idx, ano in enumerate(anos, start=1):
    df_ano = df_ocorrencia_recente[df_ocorrencia_recente['ocorrencia_dia'].dt.year == ano]
    for col_idx, cls in enumerate(classificacoes, start=1):
        contagens = (
            df_ano[df_ano['ocorrencia_classificacao'] == cls]
            .groupby('ocorrencia_cidade').size()
            .reset_index(name='contagem')
            .sort_values('contagem', ascending=False)
            .reset_index(drop=True)
        )
        contagens['rank'] = range(1, len(contagens) + 1)
        dist_cidades_ano_classificacao.add_trace(
            go.Scatter(
                x=contagens['rank'],
                y=contagens['contagem'],
                mode='markers',
                marker=dict(color=cores[cls], size=6, opacity=0.7),
                text=contagens['ocorrencia_cidade'],
                hovertemplate='<b>%{text}</b><br>Rank: %{x}<br>Ocorrências: %{y}<extra></extra>',
                showlegend=False,
            ),
            row=row_idx, col=col_idx,
        )
        show_x = row_idx == len(anos)
        show_y = col_idx == 1
        dist_cidades_ano_classificacao.update_xaxes(title_text='Rank da cidade' if show_x else '', row=row_idx, col=col_idx)
        dist_cidades_ano_classificacao.update_yaxes(title_text='Ocorrências' if show_y else '', row=row_idx, col=col_idx)

legend_text = (
    '<span style="color:#EF553B">&#9632;</span> Acidente    '
    '<span style="color:#f7a831">&#9632;</span> Incidente Grave    '
    '<span style="color:#27D3F5">&#9632;</span> Incidente'
)

dist_cidades_ano_classificacao.update_layout(
    height=700, width=1100,
    title_text='Ranking ocorrências por cidade - 2023 e 2024',
    annotations=[dict(
        text=legend_text,
        xref='paper', yref='paper',
        x=0.5, y=1.06,
        showarrow=False,
        font=dict(size=13),
        xanchor='center',
    )],
    margin=dict(t=100),
)
dist_cidades_ano_classificacao.show()

De fato, confirmamos que acidentes tem uma distribuição constante e em baixo volume, enquanto incidentes tem uma distribuição concentrada em certos hubs/clusters. Fazendo hover acima dos pontos do subplot de incidentes, podemos ver que os pontos com maior número de incidentes estão os grandes aeroportos do país. Não surpreendentemente a cidade do Rio de Janeiro figura no topo da contagem de ambos os anos, sendo a única com dois aeroportos de grande volume de movimentos. O Estado de São Paulo forma um hub maior com Congonhas, Guarulhos e Viracopos, no entanto, no conjunto de dados, a contagem de cada um desses aeroportos pertence a uma cidade diferente - e todos eles também figuram no topo do ranking de incidentes.

Aqui temos vista de que a hipótese H2 é verdadeira. De fato, Incidentes apresentam maior concentração em determinados aeroportos, enquanto acidentes e incidentes graves possuem distribuição mais uniforme.

## Distribuição das ocorrências por fase de voo
O dataframe ```df_aeronave``` contém outros atributos de interesse à este estudo. Ele mantem os dados do aeroporto de onde o voo decolou, o aeroporto onde estava programado o pouso e a fase de voo em que a ocorrência foi reportada. Num procedimento de avaliação de risco, é levado em conta as fases mais críticas de operação (STOLZER, 2016). Na aviação, as fases de decolagem, aproximação final e pouso são consideradas as mais críticas do voo, concentrando a maior parte das ocorrências aeronáuticas. Isso se deve à maior complexidade operacional nessas etapas, bem como à menor margem de erro associada à proximidade do solo.

Estudos internacionais indicam que a maioria dos acidentes ocorre durante as fases de aproximação e pouso, seguidas pela decolagem (ICAO, 2023; Boeing, 2022). Essa evidência reforça a importância de analisar as ocorrências considerando a fase do voo, como forma de compreender melhor os fatores de risco envolvidos.

Vamos observar o agrupamento das ocorrências registradas por fase de voo para obter a confirmação do embasamento teórico citado.

In [9]:
#agrupamento de ocorrencia por faze de voo
df_ocorrencia_fase_voo = df_aeronave[df_aeronave['aeronave_fase_operacao'].notna()]

df_ocorrencia_com_fase = df_ocorrencia.merge(
    df_ocorrencia_fase_voo[['codigo_ocorrencia2', 'aeronave_fase_operacao']],
    on='codigo_ocorrencia2',
    how='inner'
)
df_ocorrencia_com_fase_2023 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2023])
df_ocorrencia_com_fase_2024 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2024])

contagem_ocorrencias_por_fase_2023 = (
    df_ocorrencia_com_fase_2023
    .groupby('aeronave_fase_operacao')
    .size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

contagem_ocorrencias_por_fase_2024 = (
    df_ocorrencia_com_fase_2024
    .groupby('aeronave_fase_operacao')
    .size()
    .reset_index(name='contagem')
    .sort_values('contagem', ascending=False)
)

print("2023")
display(contagem_ocorrencias_por_fase_2023.head())
print("2024")
display(contagem_ocorrencias_por_fase_2024.head())

2023


,aeronave_fase_operacao,contagem
18,POUSO,451
8,DECOLAGEM,353
7,CRUZEIRO,151
1,APROXIMAÇÃO FINAL,143
22,TÁXI,56


2024


,aeronave_fase_operacao,contagem
17,POUSO,618
5,CRUZEIRO,551
6,DECOLAGEM,452
20,REVISÃO DE PISTA,265
0,APROXIMAÇÃO FINAL,243


## Distribuição das ocorrências por fase de voo dentro de cada aeroporto

Para definir em qual aeroporto de fato se deu a ocorrência, ainda usando o dataframe ```df_aeronave``` precisamos mapear a qual fase de voo em que cada aeroporto está implicado. Com esse fim, foi criado map que direciona a fase do voo para a ponta onde ela ocorreu. Decolagem e Pousos são exemplos claros de que os aeroportos implicados correspondem, respectivamente, aos atributos aeronave_voo_origem e aeronave_voo_destino. No entando, em alguns casos não há um aeroporto relacionado - aeronave em cruzeiro, por exemplo - ou não podemos identificar claramente qual a ponta da ocorrência: táxi e manobra são exemplos. Quando não for possível determinar o provável local da ocorrência, o a contagem é marcada como NAO IDENTIFICADO

Tamém, este agrupamento será utilizado em análises posteriores. O CENIPA registra as ocorrências mantendo apenas o nome do aeroporto, enquanto a ANAC utiliza o ICAO como identificador único. Nesse caso, esse agrupamento irá facilitar o mapeamento de correspondência entre ICAO e nome do aeroporto.

In [10]:
df_ocorrencia_fase_voo = df_aeronave[df_aeronave['aeronave_fase_operacao'].notna()]

df_ocorrencia_com_fase = df_ocorrencia.merge(
    df_ocorrencia_fase_voo[['codigo_ocorrencia2', 'aeronave_fase_operacao']],
    on='codigo_ocorrencia2',
    how='inner'
)
df_ocorrencia_com_fase_2023 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2023])
df_ocorrencia_com_fase_2024 = filter_by_year(df_ocorrencia_com_fase, 'ocorrencia_dia', [2024])

print("2023:", df_ocorrencia_com_fase_2023.shape)
print("2024:", df_ocorrencia_com_fase_2024.shape)

contagem_ocorrencias_aeroporto_2023 = build_ocorrencias_por_aeroporto(df_ocorrencia_com_fase_2023, df_aeronave)
contagem_ocorrencias_aeroporto_2024 = build_ocorrencias_por_aeroporto(df_ocorrencia_com_fase_2024, df_aeronave)

print("2023")
display(contagem_ocorrencias_aeroporto_2023)
print("2024")
display(contagem_ocorrencias_aeroporto_2024)

2023: (1396, 23)
2024: (2701, 23)
2023


,aeroporto,ocorrencia_classificacao,contagem
0,VIRACOPOS,INCIDENTE,73
1,GOVERNADOR ANDRÉ FRANCO MONTORO,INCIDENTE,71
2,CONGONHAS,INCIDENTE,64
3,SALGADO FILHO,INCIDENTE,61
4,PRESIDENTE JUSCELINO KUBITSCHEK,INCIDENTE,47
...,...,...,...
202,FAZENDA FORTALEZA DO GUAPORÉ,INCIDENTE,1
203,FAZENDA IROHY,ACIDENTE,1
204,FAZENDA NOSSA SENHORA DE FÁTIMA,ACIDENTE,1
205,FAZENDA TRADIÇÃO,INCIDENTE,1


2024


,aeroporto,ocorrencia_classificacao,contagem
0,GOVERNADOR ANDRÉ FRANCO MONTORO,INCIDENTE,145
1,ANTONIO CARLOS JOBIM / GALEÃO,INCIDENTE,93
2,CONGONHAS,INCIDENTE,72
3,GUARARAPES - GILBERTO FREYRE,INCIDENTE,72
4,PRESIDENTE JUSCELINO KUBITSCHEK,INCIDENTE,65
...,...,...,...
222,JATAÍ,INCIDENTE,1
223,BAURU,ACIDENTE,1
224,JOÃO MONTEIRO,INCIDENTE,1
225,Comandante Rolim Adolfo Amaro,INCIDENTE GRAVE,1


## Análise dos dados do Registro de Voos Ativos (VRA/ANAC)

Conforme brevemente abordado na sessão anterior, existem parâmetros a se considerar durante a avaliação do risco de uma operação crítica. Nesta abordagem, a avaliação da segurança operacional deve considerar não apenas o número de ocorrências, mas também o nível de exposição do sistema às operações aeronáuticas.

De acordo com a International Civil Aviation Organization (ICAO, 2018), indicadores de segurança devem ser normalizados por medidas de atividade, como número de voos ou horas de voo, de forma a refletir adequadamente o risco operacional. De maneira complementar, a Federal Aviation Administration (FAA, 2016) destaca que o risco é função não apenas da probabilidade e severidade de eventos, mas também da exposição ao sistema.

Até aqui tratamos com informações quantitativas. A partir de agora **mantenha o cinto de segurança afivelado até que o sinal luminoso seja apagado**. Vamos dar mais um passo para a verificação das hipóteses H1 e H3.

### Tratamento dos dados: Contagem de movimentos por aeroporto
Nos dataframes ```df_vra_2023```  e  ```df_vra_2024``` estão registrados todos os voos programados para todos aeroportos no Brasil. Para fazer a contagem de movimentos, foram consideradas as linhas onde a Situação do Voo = ```REALIZADO```. Para os aeroportos que recebem e realizam voos internacionais, contabilizamos apenas o movimento ocorrido aeroporto do Brasil para manter os dados concisos de acordo com o objetivo do estudo.

O número de movimentos por aeroporto será o nosso fator exposição.





In [11]:
#contagem de movimentos VRA por aeroporto indexada por sigla ICAO (para cruzamento com CENIPA)
# Sigla ICAO é um identificador único para cada aeroporto no mundo, registrado na International Civil Aviation Organization (ICAO)

def _group_VRA_BR_icao(df):
    from collections import defaultdict
    group_by_icao = defaultdict(lambda: {"pousos": 0, "decolagens": 0})

    ICAO_ORIGEM  = "Sigla ICAO Aeroporto Origem"
    ICAO_DESTINO = "Sigla ICAO Aeroporto Destino"
    DESC_ORIGEM  = "Descrição Aeroporto Origem"
    DESC_DESTINO = "Descrição Aeroporto Destino"
    SITUACAO_VOO = "Situação Voo"

    cols = list(df.columns)
    i_sit       = cols.index(SITUACAO_VOO)
    i_icao_orig = cols.index(ICAO_ORIGEM)
    i_icao_dest = cols.index(ICAO_DESTINO)
    i_desc_orig = cols.index(DESC_ORIGEM)
    i_desc_dest = cols.index(DESC_DESTINO)

    for row in df.itertuples(index=False, name=None):
        if row[i_sit] != "REALIZADO":
            continue

        desc_orig = row[i_desc_orig]
        desc_dest = row[i_desc_dest]
        icao_orig = row[i_icao_orig]
        icao_dest = row[i_icao_dest]

        if pd.notna(desc_orig) and "brasil" in str(desc_orig).lower() and pd.notna(icao_orig):
            group_by_icao[str(icao_orig).strip()]["decolagens"] += 1
        if pd.notna(desc_dest) and "brasil" in str(desc_dest).lower() and pd.notna(icao_dest):
            group_by_icao[str(icao_dest).strip()]["pousos"] += 1

    out = pd.DataFrame.from_dict(group_by_icao, orient="index")
    out["movimentacao_total"] = out["pousos"] + out["decolagens"]
    return out.sort_values("movimentacao_total", ascending=False)

contagem_movimentos_br_icao_2023 = _group_VRA_BR_icao(df_vra_2023)
contagem_movimentos_br_icao_2024 = _group_VRA_BR_icao(df_vra_2024)

print("2023:", contagem_movimentos_br_icao_2023.shape)
display(contagem_movimentos_br_icao_2023.head(10))
print("2024:", contagem_movimentos_br_icao_2024.shape)
display(contagem_movimentos_br_icao_2024.head(10))

2023: (191, 3)


,pousos,decolagens,movimentacao_total
SBGR,131339,131391,262730
SBSP,92987,92991,185978
SBKP,62075,62083,124158
SBBR,55304,55307,110611
SBRJ,53344,53365,106709
SBCF,48534,48535,97069
SBRF,38159,38091,76250
SBPA,30534,30525,61059
SBSV,27606,27596,55202
SBCT,26446,26656,53102


2024: (191, 3)


,pousos,decolagens,movimentacao_total
SBGR,136722,136735,273457
SBSP,94257,94262,188519
SBKP,59851,59916,119767
SBCF,55894,55899,111793
SBBR,54516,54509,109025
SBGL,48762,48785,97547
SBRF,41143,41092,82235
SBRJ,28583,28620,57203
SBSV,27653,27645,55298
SBCT,25953,26162,52115


| ICAO | Nome do Aeroporto | Pousos | Decolagens | Movimentação Total |
|------|-------------------|--------|------------|-------------------|
| SBGR | Aeroporto Internacional de Guarulhos (São Paulo) | 131.339 | 131.391 | 262.730 |
| SBSP | Aeroporto de Congonhas (São Paulo) | 92.987 | 92.991 | 185.978 |
| SBKP | Aeroporto Internacional de Viracopos (Campinas) | 62.075 | 62.083 | 124.158 |
| SBBR | Aeroporto Internacional de Brasília | 55.304 | 55.307 | 110.611 |
| SBRJ | Aeroporto Santos Dumont (Rio de Janeiro) | 53.344 | 53.365 | 106.709 |
| SBCF | Aeroporto Internacional Tancredo Neves (Confins/BH) | 48.534 | 48.535 | 97.069 |
| SBRF | Aeroporto Internacional do Recife/Guararapes | 38.159 | 38.091 | 76.250 |
| SBPA | Aeroporto Internacional Salgado Filho (Porto Alegre) | 30.534 | 30.525 | 61.059 |
| SBSV | Aeroporto Internacional Dep. Luís Eduardo Magalhães (Salvador) | 27.606 | 27.596 | 55.202 |
| SBCT | Aeroporto Internacional Afonso Pena (Curitiba) | 26.446 | 26.656 | 53.102 |

## Cálculo do índice de exposição ao risco

Neste ponto, temos o risco e a exposição. Com total de movimentos por aeroporto e o total de ocorrências registrado por aeroporto, agrupado pela classificação, podemos calcular os índices de risco normalizado por número de voos ao invés de considerar números absolutos.

### Tratamento de dados: _Fuzzy Matching_ e join dos dados ANAC e CENIPA

Em sessões anteriores, descobrimos que as bases de dados da ANAC e do CENIPA não possuem um atributo único comum que permita um join direto dos dois dataframes. Para possibilitar o cruzamento dos dados, são executadas as ações:

1 - Preparação ANAC — combina os dois anos de VRA e constrói dois dicionários: um para lookup normalizado (usado no match) e outro para nome de exibição limpo.

2 - Match CENIPA → ICAO — para cada aeroporto CENIPA, normaliza o nome, separa partes compostas por / e pontua candidatos por substring, escolhendo o de maior score.

3 - Unificação — a coluna icao é adicionada ao DataFrame CENIPA, permitindo o join com os dados ANAC; não mapeados são logados para revisão.

Após o processamento, alguns aeroportos listados nos dados vindos do CENIPA não obtêm matches relevantes com ICAO. Os aeroportos que não puderam ser mapeados são impressos ao final da célula para inspeção.

In [12]:
#mapeamento de nomes de aeroporto CENIPA → ICAO via descrição VRA
#CENIPA usa nomes completos (ex: "VIRACOPOS"), VRA usa codigos ICAO (ex: "SBKP")
#a ponte é a coluna de descrição VRA, que contem o nome do aeroporto embutido

import unicodedata

def _normalize_str(s):
    """Remove acentos e converte para maiusculas."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(s).upper())
        if unicodedata.category(c) != 'Mn'
    )

def _build_icao_lookup(df_vra):
    """Constrói dict ICAO → descrição normalizada, filtrado para aeroportos brasileiros."""
    icao_desc = {}
    for col_icao, col_desc in [
        ('Sigla ICAO Aeroporto Origem',  'Descrição Aeroporto Origem'),
        ('Sigla ICAO Aeroporto Destino', 'Descrição Aeroporto Destino'),
    ]:
        pairs = df_vra[[col_icao, col_desc]].dropna().drop_duplicates()
        mask = pairs[col_desc].str.contains('brasil', case=False, na=False)
        for icao, desc in zip(pairs.loc[mask, col_icao], pairs.loc[mask, col_desc]):
            icao = str(icao).strip()
            if len(icao) == 4 and icao not in icao_desc:
                norm = _normalize_str(desc)
                norm = norm.replace('- BRASIL', '').replace('-BRASIL', '').strip()
                icao_desc[icao] = norm
    return icao_desc

def _match_cenipa_to_icao(cenipa_name, icao_desc):
    """Encontra o ICAO para um nome de aeroporto CENIPA por correspondência de substring.

    Separa nomes compostos por '/' (ex: "ANTONIO CARLOS JOBIM / GALEÃO") e
    pontua cada ICAO candidato pelo total de caracteres correspondidos.
    """
    parts = [p.strip() for p in _normalize_str(cenipa_name).split('/')]
    best_icao, best_score = None, 0
    for icao, norm_desc in icao_desc.items():
        score = sum(len(p) for p in parts if len(p) >= 4 and p in norm_desc)
        if score > best_score:
            best_score, best_icao = score, icao
    return best_icao

def _build_icao_to_nome(df_vra):
    """Constrói dict ICAO → nome de exibição limpo (descrição VRA sem o sufixo '- Brasil')."""
    icao_nome = {}
    for col_icao, col_desc in [
        ('Sigla ICAO Aeroporto Origem',  'Descrição Aeroporto Origem'),
        ('Sigla ICAO Aeroporto Destino', 'Descrição Aeroporto Destino'),
    ]:
        pairs = df_vra[[col_icao, col_desc]].dropna().drop_duplicates()
        mask = pairs[col_desc].str.contains('brasil', case=False, na=False)
        for icao, desc in zip(pairs.loc[mask, col_icao], pairs.loc[mask, col_desc]):
            icao = str(icao).strip()
            if len(icao) == 4 and icao not in icao_nome:
                nome = str(desc).strip()
                for suffix in [' - Brasil', '- Brasil', ' – Brasil', '– Brasil', '-Brasil']:
                    if nome.lower().endswith(suffix.lower()):
                        nome = nome[:-len(suffix)].strip()
                        break
                icao_nome[icao] = nome
    return icao_nome

# Combina VRA 2023 e 2024 para maximizar cobertura do mapeamento
_vra_combined = pd.concat([df_vra_2023, df_vra_2024], ignore_index=True)
_icao_lookup   = _build_icao_lookup(_vra_combined)
_icao_to_nome  = _build_icao_to_nome(_vra_combined)

# Aplica o mapeamento: adiciona coluna 'icao' nas tabelas de ocorrências CENIPA
for df_ocorr, label in [
    (contagem_ocorrencias_aeroporto_2023, '2023'),
    (contagem_ocorrencias_aeroporto_2024, '2024'),
]:
    df_ocorr['icao'] = df_ocorr['aeroporto'].apply(
        lambda n: _match_cenipa_to_icao(n, _icao_lookup)
    )
    mapped = df_ocorr['icao'].notna().sum()
    print(f"{label}: {mapped}/{len(df_ocorr)} registros mapeados para ICAO")

unmapped = contagem_ocorrencias_aeroporto_2023.loc[
    contagem_ocorrencias_aeroporto_2023['icao'].isna(), 'aeroporto'
].unique()
if len(unmapped):
    print(f"\nNão mapeados (2023): {unmapped}")

2023: 111/207 registros mapeados para ICAO
2024: 114/227 registros mapeados para ICAO

Não mapeados (2023): <StringArray>
[                   'TENENTE-CORONEL AVIADOR CÉSAR BOMBONATO',
                                   'AERÓDROMO NÃO CADASTRADO',
                                        'CAMPO DE MARTE - SP',
                                          'FORA DE AERODROMO',
                                'AEROCLUBE DE SANTA CATARINA',
                              'Comandante Rolim Adolfo Amaro',
                        'ÁREA DE POUSO PARA USO AEROAGRÍCOLA',
 'ESTADUAL DE CAMPOS DOS AMARAIS - PREFEITO FRANCISCO AMARAL',
                                           'NÃO IDENTIFICADO',
                                     'JOÃO SIMÕES LOPES NETO',
                                              'JOÃO MONTEIRO',
                                             'POUSO DA ÁGUIA',
                                                    'RECREIO',
                          'CAVU - CLUBE DE AVIAÇÃO ULTRALEV

### Cálculo dos índices de risco

Enfim, temos os dados prontos calcular o a taxa de risco.
A taxa de risco operacional pode ser definida como a razão entre o número de ocorrências aeronáuticas e o volume de operações realizadas em um determinado aeroporto. Considerando o número de movimentos (pousos e decolagens) como medida de exposição, a taxa pode ser expressa como $$ R = \frac{N}{M} $$
sendo frequentemente normalizada para fins comparativos (ICAO, 2018; IATA, 2023).

Essa abordagem está alinhada ao conceito de risco que considera a exposição operacional como um de seus componentes fundamentais (FAA, 2016).

In [13]:
#cruzamento CENIPA x VRA: calculo de taxa de ocorrencia por aeroporto
#join feito via coluna 'icao' (resolvida pelo mapeamento de nomes na celula anterior)

def _build_taxa_aeroporto(contagem_ocorrencias, contagem_movimentos):
    pivot = (
        contagem_ocorrencias.dropna(subset=['icao'])
        .pivot_table(index='icao', columns='ocorrencia_classificacao', values='contagem', aggfunc='sum')
        .fillna(0)
        .reset_index()
    )
    pivot.columns.name = None
    pivot = pivot.rename(columns={'icao': 'aeroporto'})

    for col in ['ACIDENTE', 'INCIDENTE GRAVE', 'INCIDENTE']:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot['total_grave']     = pivot['ACIDENTE'] + pivot['INCIDENTE GRAVE']
    pivot['total_incidente'] = pivot['INCIDENTE']

    df = pivot.merge(
        contagem_movimentos[['movimentacao_total']],
        left_on='aeroporto',
        right_index=True,
        how='inner'
    )

    df['taxa_grave']        = df['total_grave']       / df['movimentacao_total']
    df['taxa_incidente']    = df['total_incidente']   / df['movimentacao_total']
    df['total_ocorrencias'] = df['total_grave']       + df['total_incidente']
    df['taxa_total']        = df['total_ocorrencias'] / df['movimentacao_total']
    df['nome_aeroporto']    = df['aeroporto'].map(_icao_to_nome).fillna(df['aeroporto'])

    return df.reset_index(drop=True)

df_taxa_aeroporto_2023 = _build_taxa_aeroporto(contagem_ocorrencias_aeroporto_2023, contagem_movimentos_br_icao_2023)
df_taxa_aeroporto_2024 = _build_taxa_aeroporto(contagem_ocorrencias_aeroporto_2024, contagem_movimentos_br_icao_2024)

print("2023 — aeroportos cruzados:", len(df_taxa_aeroporto_2023))
display(df_taxa_aeroporto_2023.sort_values('taxa_total', ascending=False).head(10))
print("2024 — aeroportos cruzados:", len(df_taxa_aeroporto_2024))
display(df_taxa_aeroporto_2024.sort_values('taxa_total', ascending=False).head(10))

2023 — aeroportos cruzados: 86


,aeroporto,ACIDENTE,INCIDENTE,INCIDENTE GRAVE,total_grave,total_incidente,movimentacao_total,taxa_grave,taxa_incidente,total_ocorrencias,taxa_total,nome_aeroporto
6,SBBI,0.0,4.0,1.0,1.0,4.0,1,1.000000,4.000000,5.0,5.000000,BACACHERI - CURITIBA - PR
7,SBBP,1.0,3.0,0.0,1.0,3.0,4,0.250000,0.750000,4.0,1.000000,ESTADUAL ARTHUR SIQUEIRA - BRAGANÇA PAULISTA - SP
33,SBJF,0.0,1.0,0.0,0.0,1.0,2,0.000000,0.500000,1.0,0.500000,FRANCISCO DE ASSIS - JUIZ DE FORA - MG
85,SWTS,0.0,0.0,1.0,1.0,0.0,4,0.250000,0.000000,1.0,0.250000,TANGARÁ DA SERRA - TANGARÁ DA SERRA - MT
72,SDCO,0.0,2.0,1.0,1.0,2.0,22,0.045455,0.090909,3.0,0.136364,SOROCABA - SOROCABA - SP
76,SSBL,1.0,0.0,0.0,1.0,0.0,12,0.083333,0.000000,1.0,0.083333,BLUMENAU - BLUMENAU - SC
5,SBBH,0.0,9.0,0.0,0.0,9.0,182,0.000000,0.049451,9.0,0.049451,PAMPULHA - CARLOS DRUMMOND DE ANDRADE - BELO H...
81,SWBC,2.0,0.0,0.0,2.0,0.0,126,0.015873,0.000000,2.0,0.015873,BARCELOS - BARCELOS - AM
61,SBSJ,0.0,1.0,1.0,1.0,1.0,240,0.004167,0.004167,2.0,0.008333,PROFESSOR URBANO ERNESTO STUMPF - SÃO JOSÉ DOS...
77,SSCN,1.0,0.0,1.0,2.0,0.0,251,0.007968,0.000000,2.0,0.007968,CANELA - CANELA - RS


2024 — aeroportos cruzados: 90


,aeroporto,ACIDENTE,INCIDENTE,INCIDENTE GRAVE,total_grave,total_incidente,movimentacao_total,taxa_grave,taxa_incidente,total_ocorrencias,taxa_total,nome_aeroporto
6,SBBP,1.0,2.0,0.0,1.0,2.0,2,0.500000,1.000000,3.0,1.500000,ESTADUAL ARTHUR SIQUEIRA - BRAGANÇA PAULISTA - SP
75,SDAG,2.0,0.0,0.0,2.0,0.0,4,0.500000,0.000000,2.0,0.500000,ANGRA DOS REIS - ANGRA DOS REIS - RJ
77,SDJV,0.0,0.0,1.0,1.0,0.0,6,0.166667,0.000000,1.0,0.166667,MUNICIPAL DE SÃO JOÃO DA BOA VISTA - SÃO JOÃO ...
76,SDCO,0.0,4.0,0.0,0.0,4.0,35,0.000000,0.114286,4.0,0.114286,SOROCABA - SOROCABA - SP
81,SSBN,0.0,1.0,0.0,0.0,1.0,12,0.000000,0.083333,1.0,0.083333,BELÉM NOVO - PORTO ALEGRE - RS
78,SIXE,0.0,0.0,1.0,1.0,0.0,17,0.058824,0.000000,1.0,0.058824,AEROCLUBE DE ELDORADO DO SUL - ELDORADO DO SUL...
5,SBBH,1.0,5.0,1.0,2.0,5.0,177,0.011299,0.028249,7.0,0.039548,PAMPULHA - CARLOS DRUMMOND DE ANDRADE - BELO H...
82,SSCN,1.0,1.0,0.0,1.0,1.0,68,0.014706,0.014706,2.0,0.029412,CANELA - CANELA - RS
83,SSLT,0.0,1.0,0.0,0.0,1.0,66,0.000000,0.015152,1.0,0.015152,GAUDÊNCIO MACHADO RAMOS - ALEGRETE - RS
85,SWBC,0.0,1.0,0.0,0.0,1.0,80,0.000000,0.012500,1.0,0.012500,BARCELOS - BARCELOS - AM


## Análise dos resultados


In [14]:
#scatter log-log: movimentos x ocorrencias por aeroporto com limiares de percentil
#percentil 50 = aeroporto tipico; percentil 75 = limiar de risco elevado
#pontos acima da linha P75 sao aeroportos com taxa de ocorrencia acima do esperado para seu volume

import plotly.graph_objects as go
import numpy as np

def _scatter_loglog_taxa(df_2023, df_2024):
    df_all = pd.concat([
        df_2023.assign(ano='2023'),
        df_2024.assign(ano='2024')
    ], ignore_index=True)

    if 'nome_aeroporto' not in df_all.columns:
        df_all['nome_aeroporto'] = df_all['aeroporto']

    p50 = df_all['taxa_total'].quantile(0.50)
    p75 = df_all['taxa_total'].quantile(0.75)

    x_min = df_all['movimentacao_total'].min()
    x_max = df_all['movimentacao_total'].max()
    x_range = np.logspace(np.log10(x_min), np.log10(x_max), 200)

    fig = go.Figure()

    for ano, cor in [('2023', '#636EFA'), ('2024', '#EF553B')]:
        sub = df_all[df_all['ano'] == ano]
        fig.add_trace(go.Scatter(
            x=sub['movimentacao_total'],
            y=sub['total_ocorrencias'],
            mode='markers',
            name=ano,
            text=sub['nome_aeroporto'],
            customdata=sub['aeroporto'],
            hovertemplate='<b>%{text}</b> (%{customdata})<br>Movimentos: %{x:,.0f}<br>Ocorrências: %{y}<extra></extra>',
            marker=dict(color=cor, opacity=0.7, size=7)
        ))

    fig.add_trace(go.Scatter(
        x=x_range,
        y=p50 * x_range,
        mode='lines',
        name=f'Percentil 50 — taxa típica ({p50:.5f})',
        line=dict(color='gray', dash='dash', width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=x_range,
        y=p75 * x_range,
        mode='lines',
        name=f'Percentil 75 — risco elevado ({p75:.5f})',
        line=dict(color='orange', dash='dot', width=2)
    ))

    fig.update_layout(
        title='Movimentos vs Ocorrências por Aeroporto (escala log-log)',
        xaxis=dict(title='Total de Movimentos (log)', type='log'),
        yaxis=dict(title='Total de Ocorrências (log)', type='log'),
        legend=dict(orientation='h', y=1.08, x=0),
        height=560
    )

    return fig

fig_scatter_loglog = _scatter_loglog_taxa(df_taxa_aeroporto_2023, df_taxa_aeroporto_2024)
fig_scatter_loglog.show()

### Escala log-log vs Matriz de Correlação
Uma matriz de correlação entre total_movimentos e total_ocorrencias por aeroporto produziria um r positivo e significativo — mas esse resultado é trivial: aeroportos mais movimentados têm mais ocorrências simplesmente pela maior exposição (efeito de base rate). A correlação não distingue entre "este aeroporto é proporcionalmente perigoso" e "este aeroporto é simplesmente grande".

O scatter log-log responde à pergunta correta: a relação entre movimentos e ocorrências é proporcional? Em escala log-log, uma linha com inclinação 1 representa taxa constante. Pontos acima dessa linha têm taxa maior do que o esperado para seu volume; pontos abaixo têm taxa menor. O desvio em relação à linha de referência é o sinal real de segurança relativa.

Observando o gráfico notamos exatamente o cenário descrito: um pequeno grupo com volume baixo de movimentos acima da linha de referência. O grupo de grandes hubs aereos se encontram abaixo das linhas de percentil 50 e percentil 75. Com isso, temo H1 também como verdadeira. O número de ocorrências aeronáuticas não cresce proporcionalmente ao volume de operações, apresentando comportamento sublinear quando ajustado por exposição.

### Percentis como limiar de referência

Usar a média nacional como linha de referência no scatter reproduziria o mesmo problema já identificado na análise dos movimentos VRA: a distribuição de tamanho de aeroportos é fortemente assimétrica à direita, com poucos hubs grandes puxando a média para cima. Uma linha de referência baseada na média faria a maioria dos aeroportos pequenos cair abaixo dela por construção matemática — não por mérito de segurança.

Os limiares escolhidos são o percentil 50 (aeroporto típico — taxa mediana, robusta à assimetria) e o percentil 75 (limiar de risco elevado). Aeroportos acima da linha P75 têm taxa de ocorrência no quartil superior, independentemente de seu volume de tráfego.

Abaixo, as estatísticas a respeito do número de ocorrências corroboram com essa afirmação. A média do numero de incidentes mostra que muitos aeroportos pequenos estariam abaixo da linha e não por isso seriam mais seguros

In [15]:
media_acidentes = df_acidentes_cidade['contagem'].mean()
media_incidentes = df_incidentes_cidade['contagem'].mean()
media_incidentes_graves = df_incidentes_graves_cidade['contagem'].mean()

mediana_acidentes = df_acidentes_cidade['contagem'].median()
mediana_incidentes = df_incidentes_cidade['contagem'].median()
mediana_incidentes_graves = df_incidentes_graves_cidade['contagem'].median()

std_acidentes = df_acidentes_cidade['contagem'].std()
std_incidentes = df_incidentes_cidade['contagem'].std()
std_incidentes_graves = df_incidentes_graves_cidade['contagem'].std()

print(f"Média de acidentes:                 {media_acidentes:.2f}")
print(f"Mediana de acidentes:               {mediana_acidentes:.2f}")
print(f"Desvio padrão de acidentes:         {std_acidentes:.2f}")

print(f"Média de incidentes:                {media_incidentes:.2f}")
print(f"Mediana de incidentes:              {mediana_incidentes:.2f}")
print(f"Desvio padrão de incidentes:        {std_incidentes:.2f}")

print(f"Média de incidentes graves:         {media_incidentes_graves:.2f}")
print(f"Mediana de incidentes graves:       {mediana_incidentes_graves:.2f}")
print(f"Desvio padrão de incidentes graves: {std_incidentes_graves:.2f}")

Média de acidentes:                 1.30
Mediana de acidentes:               1.00
Desvio padrão de acidentes:         0.62
Média de incidentes:                17.75
Mediana de incidentes:              2.00
Desvio padrão de incidentes:        49.37
Média de incidentes graves:         1.21
Mediana de incidentes graves:       1.00
Desvio padrão de incidentes graves: 0.58


In [16]:
#voos por ocorrencia nos 5 aeroportos com maior volume de movimentos (escala inversa)
#leitura: a cada X voos, ocorre 1 evento — quanto maior o numero, mais seguro o aeroporto
#contraponto ao ranking por taxa: mostra que os grandes hubs tem numeros muito mais altos (mais seguros por voo)

def _bar_hubs_inverso(df, ano):
    top = df.nlargest(5, 'movimentacao_total').copy()
    if 'nome_aeroporto' not in top.columns:
        top['nome_aeroporto'] = top['aeroporto']

    top['voos_por_grave']     = (1 / top['taxa_grave']).replace([np.inf], pd.NA)
    top['voos_por_incidente'] = (1 / top['taxa_incidente']).replace([np.inf], pd.NA)

    df_melted = top.melt(
        id_vars=['aeroporto', 'nome_aeroporto', 'movimentacao_total'],
        value_vars=['voos_por_grave', 'voos_por_incidente'],
        var_name='tipo',
        value_name='voos_por_ocorrencia'
    )
    df_melted['tipo'] = df_melted['tipo'].map({
        'voos_por_grave':     'Graves (Acidentes + Inc. Graves)',
        'voos_por_incidente': 'Incidentes',
    })
    df_melted['rotulo'] = df_melted['voos_por_ocorrencia'].apply(
        lambda x: f'1 em {x:,.0f} voos' if pd.notna(x) else 'sem ocorrências'
    )

    order = top.sort_values('movimentacao_total', ascending=True)['nome_aeroporto'].tolist()

    fig = ptex.bar(
        df_melted.dropna(subset=['voos_por_ocorrencia']),
        x='voos_por_ocorrencia',
        y='nome_aeroporto',
        color='tipo',
        orientation='h',
        barmode='group',
        text='rotulo',
        title=f'{ano} — Intervalo médio entre ocorrências (5 aeroportos mais movimentados)',
        labels={
            'voos_por_ocorrencia': 'Voos por ocorrência  ·  maior = mais seguro',
            'nome_aeroporto': 'Aeroporto',
            'tipo': 'Tipo de ocorrência',
        },
        color_discrete_map={
            'Graves (Acidentes + Inc. Graves)': '#EF553B',
            'Incidentes': '#27D3F5',
        },
        category_orders={'nome_aeroporto': order},
        hover_data={'aeroporto': True, 'movimentacao_total': ':,.0f'},
        height=420
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_layout(
        legend=dict(orientation='h', y=1.12, x=0),
        margin=dict(r=120),
    )
    return fig

fig_hubs_2023 = _bar_hubs_inverso(df_taxa_aeroporto_2023, '2023')
fig_hubs_2024 = _bar_hubs_inverso(df_taxa_aeroporto_2024, '2024')
fig_hubs_2023.show()
fig_hubs_2024.show()

### Comparando o risco operacional de grandes aeroportos frente aos aeroportos mais seguros

Nos dois gráficos acima vemos a avaliação de risco dos aeroportos mais movimentados do país nos anos de 2023 e 2024.

Nos dois gráficos abaixo, vemos a mesma avaliação de risco para os aeroportos mais seguros do Brasil, sendo comparados pela mesma medida - número de ocorrência por milhares de voo.

Com isso, podemos confirmar a H3, quando temos uma base comum de comparação, podemos fazer comparações mais assertivas a respeito da segurança operacional nos aeroportos do Brasil.

In [17]:
#top 5 aeroportos mais seguros (maior intervalo entre ocorrencias, escala inversa)
#exclui aeroportos com taxa_total == 0 (inverso indefinido) antes de ordenar

def _bar_safest(df, ano):
    candidatos = df[df['taxa_total'] > 0].copy()
    top = candidatos.nsmallest(5, 'taxa_total').copy()

    if 'nome_aeroporto' not in top.columns:
        top['nome_aeroporto'] = top['aeroporto']

    top['voos_por_grave']     = (1 / top['taxa_grave']).replace([np.inf], pd.NA)
    top['voos_por_incidente'] = (1 / top['taxa_incidente']).replace([np.inf], pd.NA)

    df_melted = top.melt(
        id_vars=['aeroporto', 'nome_aeroporto', 'movimentacao_total'],
        value_vars=['voos_por_grave', 'voos_por_incidente'],
        var_name='tipo',
        value_name='voos_por_ocorrencia'
    )
    df_melted['tipo'] = df_melted['tipo'].map({
        'voos_por_grave':     'Graves (Acidentes + Inc. Graves)',
        'voos_por_incidente': 'Incidentes',
    })
    df_melted['rotulo'] = df_melted['voos_por_ocorrencia'].apply(
        lambda x: f'1 em {x:,.0f} voos' if pd.notna(x) else 'sem ocorrências'
    )

    order = top.sort_values('taxa_total', ascending=False)['nome_aeroporto'].tolist()

    fig = ptex.bar(
        df_melted.dropna(subset=['voos_por_ocorrencia']),
        x='voos_por_ocorrencia',
        y='nome_aeroporto',
        color='tipo',
        orientation='h',
        barmode='group',
        text='rotulo',
        title=f'{ano} — Top 5 Aeroportos mais seguros por voo (menor taxa de ocorrência)',
        labels={
            'voos_por_ocorrencia': 'Voos por ocorrência  ·  maior = mais seguro',
            'nome_aeroporto': 'Aeroporto',
            'tipo': 'Tipo de ocorrência',
        },
        color_discrete_map={
            'Graves (Acidentes + Inc. Graves)': '#EF553B',
            'Incidentes': '#27D3F5',
        },
        category_orders={'nome_aeroporto': order},
        hover_data={'aeroporto': True, 'movimentacao_total': ':,.0f'},
        height=420
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_layout(
        legend=dict(orientation='h', y=1.12, x=0),
        margin=dict(r=120),
    )
    return fig

fig_safest_2023 = _bar_safest(df_taxa_aeroporto_2023, '2023')
fig_safest_2024 = _bar_safest(df_taxa_aeroporto_2024, '2024')
fig_safest_2023.show()
fig_safest_2024.show()

## Limitações do Estudo
As maiores limitações do estudo estão no fato de que o VRA é uma amostra uniforme do movimento aeroviário, estando o estudo sujeito a um viés de exposição da aviação civil. Também as simplificações do estudo consideram um mapeamento heurístico para ligar a fase de voo ao aeroporto onde a ocorrência teve origem. Um número de ocorrências não pôde ser mapeada a um aeroporto específico devido a natureza ambígua da fase (taxi, manobra, estacionamento, outra fase, etc). Outro tipo de simplificação.

Aqui tratamos de uma métrica direta e objetiva (ocorrência por milhares de voos). Porém no panorama de segurarança operacional também podem ser considerados outros fatores como manutenções da aeronave, condições metorológicas, experiência de piloto e alto volume de passageiros.

# Bibliografia
INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Management Manual (SMM). 4. ed. Montreal: ICAO, 2018.

INTERNATIONAL AIR TRANSPORT ASSOCIATION. Safety Report 2023. Montreal: IATA, 2023

CENTRO DE INVESTIGAÇÃO E PREVENÇÃO DE ACIDENTES AERONÁUTICOS. NSCA 3-13: Protocolos de Investigação de Ocorrências Aeronáuticas. Brasília: CENIPA, 2018.

STOLZER, Alan J.; HALFORD, Carl D.; GOGLIA, John J. Introduction to Safety Management Systems. 2. ed. Boca Raton: CRC Press, 2016.

INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Report 2023. Montreal: ICAO, 2023.

BOEING. Statistical Summary of Commercial Jet Airplane Accidents: Worldwide Operations 1959–2022. Seattle: Boeing, 2022.

INTERNATIONAL CIVIL AVIATION ORGANIZATION. Safety Management Manual (SMM). 4. ed. Montreal: ICAO, 2018.

FEDERAL AVIATION ADMINISTRATION. Risk Management Handbook (FAA-H-8083-2). Washington, DC: FAA, 2016.

# Bonus: Veja a taxa de risco para qualquer um dos aeroportos listados no conjunto de dados selecionado

In [19]:
import ipywidgets as widgets
from IPython.display import display, HTML

_dfs_taxa = {2023: df_taxa_aeroporto_2023, 2024: df_taxa_aeroporto_2024}

def _airport_options(year):
    df = _dfs_taxa[year].sort_values('nome_aeroporto')
    return [(f"{row.aeroporto} — {row.nome_aeroporto}", row.aeroporto)
            for row in df.itertuples()]

def _fmt(val):
    return f"{val:,.0f}".replace(",", ".")

dd_ano = widgets.Dropdown(options=[2023, 2024], value=2023, description='Ano:')
dd_aeroporto = widgets.Dropdown(options=_airport_options(2023), description='Aeroporto:')
btn_calcular = widgets.Button(description='Calcular', button_style='primary')
out = widgets.Output()

def _on_ano_change(change):
    dd_aeroporto.options = _airport_options(change['new'])

dd_ano.observe(_on_ano_change, names='value')

def _on_calcular(_):
    out.clear_output(wait=True)
    with out:
        df = _dfs_taxa[dd_ano.value]
        row = df[df['aeroporto'] == dd_aeroporto.value]
        if row.empty:
            print("Aeroporto não encontrado.")
            return
        row = row.iloc[0]

        if row.taxa_incidente > 0:
            txt_inc = f"1 ocorrência a cada <b>{_fmt(1 / row.taxa_incidente)}</b> voos"
        else:
            txt_inc = "Sem registros"

        if row.taxa_grave > 0:
            txt_grave = f"1 ocorrência a cada <b>{_fmt(1 / row.taxa_grave)}</b> voos"
        else:
            txt_grave = "Sem registros"

        display(HTML(
            f"<p style='font-size:15px; margin:4px 0'>"
            f"<span style='color:#27D3F5'>&#9632;</span> "
            f"<b>Incidentes:</b> {txt_inc}</p>"
            f"<p style='font-size:15px; margin:4px 0'>"
            f"<span style='color:#EF553B'>&#9632;</span> "
            f"<b>Acidentes + Incidentes Graves:</b> {txt_grave}</p>"
        ))

btn_calcular.on_click(_on_calcular)

widgets.VBox([widgets.HBox([dd_ano, dd_aeroporto, btn_calcular]), out])
